# Trabalho Final - Inteligência Computacional
## Classificação de países por grupo de renda usando dados climáticos

**Faculdade de Tecnologia de Jundiaí**  
Curso Superior de Tecnologia em Ciência de Dados  
Professor: Me. Mateus Guilherme Fuini

Dataset: NASA POWER Climate Risk Indices - 190 capitais (1990-2024)  
Fonte: Kaggle  
Problema: Classificação (wb_income_group)


## Importando as bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

import warnings
warnings.filterwarnings('ignore')


## Carregando o dataset

In [ ]:
df = pd.read_csv('nasa_power_climate_risk_indices_190_capitals_1990_2024.csv')
print(df.shape)
df.head()


In [ ]:
# vendo os tipos das colunas
df.dtypes


In [ ]:
# verificando valores nulos
df.isnull().sum()


In [ ]:
df.describe()


## Análise Exploratória

### Distribuição da variável alvo

In [ ]:
df['wb_income_group'].value_counts()


In [ ]:
df['wb_income_group'].value_counts().plot(kind='bar')
plt.title('Quantidade por grupo de renda')
plt.xlabel('Grupo')
plt.ylabel('Quantidade')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Histogramas das variáveis numéricas principais

In [ ]:
# escolhi algumas colunas que achei mais relevantes
cols = ['temp_mean_c', 'precip_total_mm', 'rh_mean_pct', 'wind_mean_ms', 'solar_mean_mj']

df[cols].hist(bins=30, figsize=(14, 8))
plt.suptitle('Histogramas')
plt.tight_layout()
plt.show()


### Boxplot - temperatura por grupo de renda

In [ ]:
# removendo Unknown pois não é um grupo válido
df2 = df[df['wb_income_group'] != 'Unknown']

plt.figure(figsize=(10, 5))
sns.boxplot(data=df2, x='wb_income_group', y='temp_mean_c')
plt.title('Temperatura média por grupo de renda')
plt.show()


### Scatterplot - temperatura x precipitação

In [ ]:
plt.figure(figsize=(9, 5))
grupos = df2['wb_income_group'].unique()
for g in grupos:
    sub = df2[df2['wb_income_group'] == g]
    plt.scatter(sub['temp_mean_c'], sub['precip_total_mm'], label=g, alpha=0.3, s=8)

plt.xlabel('Temperatura média (C)')
plt.ylabel('Precipitação (mm)')
plt.title('Temperatura x Precipitação')
plt.legend()
plt.show()


### Heatmap de correlação

In [ ]:
# peguei só algumas colunas pra não ficar grande demais
cols_corr = ['temp_mean_c', 'precip_total_mm', 'rh_mean_pct', 'wind_mean_ms',
             'solar_mean_mj', 'heat_stress_index', 'days_above_35c',
             'gdp_per_capita_usd', 'energy_use_kg_oil_eq']

plt.figure(figsize=(10, 8))
sns.heatmap(df2[cols_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlação entre variáveis')
plt.tight_layout()
plt.show()


## Pré-processamento

Aqui vou preparar os dados para o modelo. Preciso:
- remover a classe Unknown
- separar X e y
- tratar os valores nulos
- codificar as categóricas
- escalonar os dados


In [ ]:
# removendo Unknown
df_model = df[df['wb_income_group'] != 'Unknown'].copy()

# separando target
y = df_model['wb_income_group']

# removendo colunas que não vou usar como feature
X = df_model.drop(columns=['city', 'iso_alpha3', 'wb_income_group'])

print(X.shape)
print(y.value_counts())


In [ ]:
# separando colunas numericas e categoricas
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

print('Numéricas:', len(num_cols))
print('Categóricas:', cat_cols)


## Engenharia de atributos

Criei 2 novas colunas a partir das originais:

1. **temp_precip_ratio**: divisão entre temperatura média e precipitação total. A ideia é capturar se o clima é quente e seco ou frio e chuvoso ao mesmo tempo.

2. **climate_risk_score**: somatório de dias de calor extremo, dias de chuva forte e metade dos dias secos. Tentei criar uma variável que resuma o risco climático geral do lugar.


In [ ]:
X = X.copy()

# atributo 1: índice de aridez simplificado
X['temp_precip_ratio'] = X['temp_mean_c'] / (X['precip_total_mm'] + 1)

# atributo 2: score de risco climático
X['climate_risk_score'] = (X['days_above_35c'].fillna(0) +
                            X['days_heavy_rain'].fillna(0) +
                            X['days_dry'].fillna(0) * 0.5)

# atualizando lista de colunas numericas
num_cols = X.select_dtypes(include='number').columns.tolist()

print(X[['temp_precip_ratio', 'climate_risk_score']].head())


## Dividindo treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Treino:', X_train.shape)
print('Teste:', X_test.shape)


## Pipeline

Usei Pipeline + ColumnTransformer para organizar o fluxo e evitar data leakage.

**O que é data leakage?**  
É quando o modelo "vê" informações do conjunto de teste durante o treinamento, o que deixa a avaliação enganosa. Por exemplo, se eu calcular a média para imputar os valores nulos usando o dataset inteiro (incluindo o teste), o modelo estaria sendo treinado com informação que não deveria ter.

O Pipeline resolve isso porque o fit() só roda nos dados de treino. No teste, só o transform() é aplicado.


In [ ]:
# pipeline para colunas numericas
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# pipeline para colunas categoricas
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# juntando tudo
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# pipeline final com o modelo
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier())
])


## Validação com K-Fold

O K-Fold Cross Validation divide os dados de treino em k partes. Em cada rodada, uma parte diferente é usada como validação e o restante para treinar. Isso dá uma ideia mais confiável da performance do modelo do que simplesmente testar uma vez.

Usei k=5.


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X_train, y_train, cv=kf, scoring='accuracy')

print('Acurácia por fold:', scores.round(4))
print('Média:', scores.mean().round(4))
print('Desvio padrão:', scores.std().round(4))


## GridSearchCV - testando hiperparâmetros

O GridSearchCV testa várias combinações de parâmetros e usa cross-validation para ver qual funciona melhor.

Parâmetros testados:
- **n_neighbors**: quantos vizinhos considerar - afeta direto o resultado, poucos vizinhos = modelo instável, muitos = muito genérico
- **weights**: se todos os vizinhos têm o mesmo peso ou se os mais próximos valem mais
- **metric**: como calcular a distância entre os pontos


In [ ]:
params = {
    'model__n_neighbors': [3, 5, 7, 11, 15],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan']
}

grid = GridSearchCV(pipeline, params, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print('Melhores parâmetros:', grid.best_params_)
print('Melhor acurácia no CV:', round(grid.best_score_, 4))


## Avaliação final no conjunto de teste

In [ ]:
y_pred = grid.best_estimator_.predict(X_test)

print('Acurácia:', round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=grid.best_estimator_.classes_)
disp = ConfusionMatrixDisplay(cm, display_labels=grid.best_estimator_.classes_)
disp.plot(cmap='Blues')
plt.title('Matriz de Confusão')
plt.tight_layout()
plt.show()


## Discussão

**Dificuldades:**  
A maior dificuldade foi lidar com os valores ausentes, especialmente em colunas como `energy_use_kg_oil_eq` que tinha bastante dado faltando. Também tivemos que remover a classe 'Unknown' pois não faz sentido tentar prever uma classe indefinida.

**Impacto do pré-processamento:**  
O escalonamento foi muito importante pro KNN funcionar direito, porque ele usa distância entre os pontos e sem escalonar as variáveis com valores maiores acabam dominando o cálculo. A imputação pela mediana também ajudou bastante pois algumas variáveis tinham distribuição bem assimétrica.

**Limitações:**  
O dataset só tem dados climáticos e alguns indicadores econômicos básicos. Fatores como história, política e infraestrutura não estão representados, o que limita bastante o que o modelo consegue aprender.

**Melhorias futuras:**  
- Testar outros algoritmos como Random Forest  
- Tentar criar mais features a partir das séries temporais  
- Ver se balancear as classes melhoraria o resultado pra classe Low  
